# Modelo Baseline. Pruebas varias

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

df = pd.read_csv("imdb_reviews.csv")

## Vectorización con TF-IFD

In [13]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df["review"]).toarray()
y = df["label"].copy()

# split train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# split dev y test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

print(len(X_train_scaled), len(X_dev_scaled), len(X_test_scaled))

35000 7500 7500


## Redes densas con cantidad de neuronas decreciente. 
Intención: reducción de dimensionalidad

### Modelo Overkill. Máxima capacidad, regularización casi mínima
256 -> 128 -> 64 -> salida

In [ ]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()

c:\Github\inteligencia_artificial_tp\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Resumen del modelo de red neuronal:


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 256)            │     2,560,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,601,473 (9.92 MB)

 Trainable params: 2,601,473 (9.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 37s 32ms/step - accuracy: 0.8275 - loss: 1.7235 - val_accuracy: 0.8421 - val_loss: 1.0737
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 36s 33ms/step - accuracy: 0.8519 - loss: 1.0639 - val_accuracy: 0.8505 - val_loss: 0.9501
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 35s 32ms/step - accuracy: 0.8599 - loss: 0.9338 - val_accuracy: 0.8509 - val_loss: 0.8700
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 37s 34ms/step - accuracy: 0.8631 - loss: 0.8069 - val_accuracy: 0.8524 - val_loss: 0.7389
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 36s 33ms/step - accuracy: 0.8650 - loss: 0.7250 - val_accuracy: 0.8507 - val_loss: 0.7198
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 36s 33ms/step - accuracy: 0.8657 - loss: 0.7106 - val_accuracy: 0.8575 - val_loss: 0.7234
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 34s 31ms/step - accuracy: 0.8687 - loss: 0.6787 - val_accuracy: 0.8537 - val_loss: 0.6682
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3

Accuracy en train no sube a valores extremos en comparación al valor en dev, pero ambos se mantienen en un rango relativamente bajo desde las primeras épocas, con subidas y bajadas constantes.

In [17]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8736
Pérdida del modelo en el conjunto de prueba: 0.6077
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.89      0.85      0.87      3750
           1       0.86      0.89      0.88      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Resultado aceptable/mediocre

### Red más pequeña, regularización levemente menor (l2 0.01 -> 0.001)
128 -> 64 -> 32 -> salida

In [ ]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()


Resumen del modelo de red neuronal:


c:\Github\inteligencia_artificial_tp\.venv\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,290,497 (4.92 MB)

 Trainable params: 1,290,497 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.8338 - loss: 0.8777 - val_accuracy: 0.8700 - val_loss: 0.8181
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9065 - loss: 0.6965 - val_accuracy: 0.8657 - val_loss: 0.7443
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9155 - loss: 0.6232 - val_accuracy: 0.8653 - val_loss: 0.7114
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 21s 17ms/step - accuracy: 0.9207 - loss: 0.5899 - val_accuracy: 0.8672 - val_loss: 0.7028
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 20s 18ms/step - accuracy: 0.9256 - loss: 0.5604 - val_accuracy: 0.8593 - val_loss: 0.7252
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.9270 - loss: 0.5616 - val_accuracy: 0.8627 - val_loss: 0.7146
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 20s 17ms/step - accuracy: 0.9366 - loss: 0.5322 - val_accuracy: 0.8627 - val_loss: 0.7441
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1

Mejor ajuste de punta a punta, pero con problemas de aparente overfit.

In [20]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8695
Pérdida del modelo en el conjunto de prueba: 0.6088
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.88      0.85      0.87      3750
           1       0.86      0.89      0.87      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Resultados en test ligeramente inferiores

### Misma red, penalización levemente más fuerte:
* l2 = 0.01

In [16]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()


Resumen del modelo de red neuronal:


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,290,497 (4.92 MB)

 Trainable params: 1,290,497 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 22s 18ms/step - accuracy: 0.8261 - loss: 1.5208 - val_accuracy: 0.8384 - val_loss: 0.9295
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.8557 - loss: 0.8818 - val_accuracy: 0.8505 - val_loss: 0.8574
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 18ms/step - accuracy: 0.8595 - loss: 0.8597 - val_accuracy: 0.8555 - val_loss: 0.8212
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.8626 - loss: 0.7924 - val_accuracy: 0.8567 - val_loss: 0.7575
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.8657 - loss: 0.7418 - val_accuracy: 0.8576 - val_loss: 0.7179
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.8652 - loss: 0.7133 - val_accuracy: 0.8551 - val_loss: 0.7131
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.8671 - loss: 0.6981 - val_accuracy: 0.8559 - val_loss: 0.6875
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 1

In [18]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8672
Pérdida del modelo en el conjunto de prueba: 0.6102
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.91      0.82      0.86      3750
           1       0.83      0.92      0.87      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Sin ninguna mejora significativa respecto del anterior ajuste, a excepción de una leve baja en los valores de la función de pérdida, y una estabilización relativa en el accuracy del conjunto dev.

### Reducción de dimensionalidad con SVD

In [19]:
vectorizer = TfidfVectorizer(max_features=10000)
X = vectorizer.fit_transform(df["review"]).toarray()
y = df["label"].copy()

# Reducción de dimensionalidad con SVD
svd = TruncatedSVD(n_components=300)
X_svd = svd.fit_transform(X)

# split train + temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_svd, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# split dev y test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_dev_scaled = scaler.transform(X_dev)
X_test_scaled = scaler.transform(X_test)

print(len(X_train_scaled), len(X_dev_scaled), len(X_test_scaled))

35000 7500 7500


In [22]:
# Definir el modelo de red neuronal
model = Sequential([
    # Capa de entrada y primera capa oculta con activación ReLU y regularización L2
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],), kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout para mitigar el sobreajuste

    # Segunda capa oculta con activación ReLU y regularización L2
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Tercera capa oculta con activación ReLU y regularización L2
    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3), # Capa Dropout

    # Capa de salida con activación sigmoide para clasificación binaria
    Dense(1, activation='sigmoid')
])


model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Mostrar un resumen del modelo
print("\nResumen del modelo de red neuronal:")
model.summary()


Resumen del modelo de red neuronal:


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 32)             │         9,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,193 (78.88 KB)

 Trainable params: 20,193 (78.88 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenar el modelo
print("\nEntrenando el modelo...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=100, # Número máximo de épocas
    batch_size=32, # Tamaño del lote para el entrenamiento
    validation_data=(X_dev_scaled, y_dev), # Datos de validación
    callbacks=[early_stopping], # Aplicar Early Stopping
    verbose=1 # Mostrar progreso del entrenamiento
)

print("\nEntrenamiento completado.")


Entrenando el modelo...
Epoch 1/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.7768 - loss: 0.5771 - val_accuracy: 0.8631 - val_loss: 0.4096
Epoch 2/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.8631 - loss: 0.3955 - val_accuracy: 0.8672 - val_loss: 0.3699
Epoch 3/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8730 - loss: 0.3594 - val_accuracy: 0.8661 - val_loss: 0.3588
Epoch 4/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8771 - loss: 0.3451 - val_accuracy: 0.8715 - val_loss: 0.3526
Epoch 5/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8810 - loss: 0.3339 - val_accuracy: 0.8676 - val_loss: 0.3515
Epoch 6/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8817 - loss: 0.3332 - val_accuracy: 0.8705 - val_loss: 0.3485
Epoch 7/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8852 - loss: 0.3279 - val_accuracy: 0.8684 - val_loss: 0.3505
Epoch 8/100
1094/1094 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - a

In [24]:
# Evaluar el modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nPrecisión del modelo en el conjunto de prueba: {accuracy:.4f}")
print(f"Pérdida del modelo en el conjunto de prueba: {loss:.4f}")

# Realizar predicciones en el conjunto de prueba
y_pred_proba = model.predict(X_test_scaled)
y_pred = (y_pred_proba > 0.5).astype(int) # Convertir probabilidades a clases binarias (0 o 1)

# Mostrar el reporte de clasificación
print("\nReporte de clasificación en el conjunto de prueba:")
print(classification_report(y_test, y_pred))


Precisión del modelo en el conjunto de prueba: 0.8668
Pérdida del modelo en el conjunto de prueba: 0.3515
235/235 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

Reporte de clasificación en el conjunto de prueba:
              precision    recall  f1-score   support

           0       0.87      0.87      0.87      3750
           1       0.87      0.87      0.87      3750

    accuracy                           0.87      7500
   macro avg       0.87      0.87      0.87      7500
weighted avg       0.87      0.87      0.87      7500



Ampliamente superior en eficiencia para un performance prácticamente idéntico en accuracy, y con mejoras en los valores de pérdida.

Aún así, los resultados de este notebook probablemente esten siendo sumamente afectados porlas limitaciones de las features, producto de una vectorización básica y un NLP pobre.